# SHL Grammar Scoring Engine

**SHL Hiring Assessment 2026 — Research Intern**

This notebook implements a multimodal regression pipeline for predicting a continuous
English grammar score (0–5) from spoken-audio samples.

**Author:** Sushank Reddy

### Pipeline

`Audio → Whisper transcription → linguistic + acoustic features → regression models → ensemble → submission.csv`

> **Data access:** The competition dataset is private and is not redistributed in this repository.
> Kaggle access is required to run the data-download cells.

## 1. Problem Statement

The task is to predict a continuous grammar score for spoken English audio.
The target score ranges from **0 to 5**, where higher values indicate stronger grammatical
accuracy and control.

The training set contains **769 labeled audio samples** and the test set contains
**216 samples**.

In [ ]:
# Install the main dependencies in Google Colab.
# Kaggle access to the private competition is required separately.

!pip -q install librosa soundfile pandas numpy scipy scikit-learn matplotlib tqdm spacy faster-whisper
!pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12
!python -m spacy download en_core_web_sm

In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.stats import pearsonr

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import mean_squared_error

import spacy
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Download the Private Kaggle Dataset

Authenticate with Kaggle using your own credentials before running the download command.
**Never commit `kaggle.json`, access tokens, or other credentials to GitHub.**

In [ ]:
# Run this after Kaggle CLI authentication.
# The competition is private, so a valid invitation/access is required.

DATA_ROOT = "/content/SHL_dataset"
DATA_DIR = os.path.join(DATA_ROOT, "Dataset_Final")
ZIP_DIR = "/content/SHL_data"

os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

# Uncomment when authenticated:
# !kaggle competitions download -c shl-hiring-assessment-2026 -p /content/SHL_data
# !unzip -qo /content/SHL_data/shl-hiring-assessment-2026.zip -d /content/SHL_dataset

print("Expected data directory:", DATA_DIR)

In [ ]:
train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_df.shape)

display(train_df.head())

## 3. Audio Transcription with Faster-Whisper

The training and test audio are transcribed using **Faster-Whisper (`small.en`)**.
The successful Colab workflow used 16 kHz mono audio loaded with `librosa`, followed by
Whisper transcription with English language detection and VAD filtering.

The transcription is then converted into linguistic features.

In [ ]:
# Make NVIDIA libraries visible to CTranslate2 in Colab when using GPU.
cuda_lib_paths = glob.glob(
    "/usr/local/lib/python*/dist-packages/nvidia/*/lib"
)

if cuda_lib_paths:
    os.environ["LD_LIBRARY_PATH"] = (
        ":".join(cuda_lib_paths)
        + ":"
        + os.environ.get("LD_LIBRARY_PATH", "")
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    "small.en",
    device=device,
    compute_type=compute_type
)

print("Whisper device:", device)
print("Compute type:", compute_type)

In [ ]:
TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "train")
TEST_AUDIO_DIR = os.path.join(DATA_DIR, "test")
ARTIFACT_DIR = "/content/shl_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

TRAIN_TRANSCRIPT_PATH = os.path.join(ARTIFACT_DIR, "train_transcripts.csv")
TEST_TRANSCRIPT_PATH = os.path.join(ARTIFACT_DIR, "test_transcripts.csv")

def transcribe_audio(audio_path):
    audio, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    segments, info = whisper_model.transcribe(
        audio,
        beam_size=5,
        language="en",
        vad_filter=True
    )

    transcript = " ".join(
        segment.text.strip()
        for segment in segments
    ).strip()

    return transcript

In [ ]:
# Transcribe the training set. Checkpoints are written every 50 files.

if os.path.exists(TRAIN_TRANSCRIPT_PATH):
    train_transcripts = pd.read_csv(TRAIN_TRANSCRIPT_PATH)
    print("Loaded cached training transcripts:", train_transcripts.shape)
else:
    results = []
    start_time = time.time()

    for i, row in enumerate(
        tqdm(train_df.itertuples(index=False), total=len(train_df),
              desc="Transcribing training audio"),
        start=1
    ):
        try:
            audio_path = os.path.join(TRAIN_AUDIO_DIR, row.filename)
            text = transcribe_audio(audio_path)
            error = None
        except Exception as e:
            text = ""
            error = str(e)

        results.append({
            "filename": row.filename,
            "label": row.label,
            "transcript": text,
            "error": error
        })

        if i % 50 == 0:
            pd.DataFrame(results).to_csv(TRAIN_TRANSCRIPT_PATH, index=False)

    train_transcripts = pd.DataFrame(results)
    train_transcripts.to_csv(TRAIN_TRANSCRIPT_PATH, index=False)

    print(f"Training transcription time: {(time.time()-start_time)/60:.2f} minutes")

print("Training transcripts:", train_transcripts.shape)
print("Transcription errors:", train_transcripts["error"].notna().sum())
display(train_transcripts.head())

## 4. Acoustic Feature Engineering

The acoustic feature extractor is kept identical for training and test data.

Features include:

- duration
- RMS energy statistics
- zero-crossing rate
- spectral centroid/bandwidth/rolloff statistics
- spectral contrast statistics
- 13 MFCC means and standard deviations
- chroma statistics
- active/silence ratios

In [ ]:
def extract_audio_features(file_path):
    y, sr = librosa.load(
        file_path,
        sr=16000,
        mono=True
    )

    features = {}

    # Duration
    features["duration"] = len(y) / sr

    # RMS
    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)
    features["rms_min"] = np.min(rms)
    features["rms_max"] = np.max(rms)

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # Spectral centroid
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features["spectral_centroid_mean"] = np.mean(spectral_centroid)
    features["spectral_centroid_std"] = np.std(spectral_centroid)

    # Spectral bandwidth
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features["spectral_bandwidth_mean"] = np.mean(spectral_bandwidth)
    features["spectral_bandwidth_std"] = np.std(spectral_bandwidth)

    # Spectral rolloff
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features["spectral_rolloff_mean"] = np.mean(spectral_rolloff)
    features["spectral_rolloff_std"] = np.std(spectral_rolloff)

    # Spectral contrast
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(spectral_contrast.shape[0]):
        features[f"spectral_contrast_{i}_mean"] = np.mean(spectral_contrast[i])
        features[f"spectral_contrast_{i}_std"] = np.std(spectral_contrast[i])

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(mfcc.shape[0]):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features["chroma_mean"] = np.mean(chroma)
    features["chroma_std"] = np.std(chroma)

    # Activity / silence
    threshold = 0.01
    active_ratio = np.mean(np.abs(y) > threshold)
    features["active_ratio"] = active_ratio
    features["silence_ratio"] = 1 - active_ratio

    return features

In [ ]:
train_acoustic_path = os.path.join(ARTIFACT_DIR, "train_acoustic_features.csv")

if os.path.exists(train_acoustic_path):
    train_acoustic = pd.read_csv(train_acoustic_path)
    print("Loaded cached acoustic features:", train_acoustic.shape)
else:
    acoustic_rows = []

    for filename in tqdm(
        train_df["filename"],
        desc="Extracting training acoustic features"
    ):
        features = extract_audio_features(
            os.path.join(TRAIN_AUDIO_DIR, filename)
        )
        features["filename"] = filename
        acoustic_rows.append(features)

    train_acoustic = pd.DataFrame(acoustic_rows)
    train_acoustic.to_csv(train_acoustic_path, index=False)

print("Training acoustic features:", train_acoustic.shape)
display(train_acoustic.head())

## 5. Linguistic Feature Engineering

The Whisper transcript is processed with **spaCy `en_core_web_sm`**.

Features capture vocabulary richness, sentence structure, repetition, stopword usage,
pronoun usage, punctuation, dependency complexity, conjunctions, bigram diversity,
and POS distributions.

In [ ]:
nlp = spacy.load("en_core_web_sm")

FILLER_WORDS = {
    "uh", "um", "hmm", "erm",
    "like", "you know",
    "actually", "basically",
    "sort of", "kind of"
}

PRONOUNS = {
    "i", "me", "my", "mine", "myself",
    "we", "us", "our", "ours",
    "you", "your", "yours",
    "he", "him", "his",
    "she", "her", "hers",
    "it", "its",
    "they", "them", "their", "theirs"
}

def extract_linguistic_features(text):
    if pd.isna(text):
        text = ""

    text = str(text).strip()
    doc = nlp(text)

    words = [
        token.text.lower()
        for token in doc
        if token.is_alpha
    ]

    word_count = len(words)
    unique_word_count = len(set(words))

    type_token_ratio = (
        unique_word_count / word_count
        if word_count > 0 else 0
    )

    avg_word_length = (
        np.mean([len(w) for w in words])
        if words else 0
    )

    sentences = list(doc.sents)
    sentence_count = len(sentences)

    sentence_lengths = [
        len([
            token for token in sent
            if token.is_alpha
        ])
        for sent in sentences
    ]

    if sentence_lengths:
        avg_sentence_length = np.mean(sentence_lengths)
        sentence_length_std = np.std(sentence_lengths)
        max_sentence_length = max(sentence_lengths)
        min_sentence_length = min(sentence_lengths)
    else:
        avg_sentence_length = 0
        sentence_length_std = 0
        max_sentence_length = 0
        min_sentence_length = 0

    pos_counts = {}
    for token in doc:
        if token.is_alpha:
            pos_counts[token.pos_] = pos_counts.get(token.pos_, 0) + 1

    def pos_ratio(pos):
        return (
            pos_counts.get(pos, 0) / word_count
            if word_count > 0 else 0
        )

    filler_count = sum(1 for word in words if word in FILLER_WORDS)
    filler_ratio = filler_count / word_count if word_count > 0 else 0

    word_counts = {}
    for word in words:
        word_counts[word] = word_counts.get(word, 0) + 1

    repeated_word_count = sum(
        count - 1
        for count in word_counts.values()
        if count > 1
    )
    repetition_ratio = (
        repeated_word_count / word_count
        if word_count > 0 else 0
    )

    stopword_count = sum(
        1 for token in doc
        if token.is_alpha and token.is_stop
    )
    stopword_ratio = (
        stopword_count / word_count
        if word_count > 0 else 0
    )

    pronoun_count = sum(
        1 for word in words
        if word in PRONOUNS
    )
    pronoun_ratio = (
        pronoun_count / word_count
        if word_count > 0 else 0
    )

    comma_count = sum(1 for token in doc if token.text == ",")
    question_count = sum(1 for token in doc if token.text == "?")
    exclamation_count = sum(1 for token in doc if token.text == "!")

    dependency_count = sum(
        1 for token in doc
        if token.dep_ not in {"ROOT", "punct"}
    )

    subordinate_count = sum(
        1 for token in doc
        if token.dep_ == "mark"
    )

    conjunction_count = sum(
        1 for token in doc
        if token.pos_ == "CCONJ"
    )

    if len(words) >= 2:
        bigrams = list(zip(words[:-1], words[1:]))
        bigram_diversity = (
            len(set(bigrams)) / len(bigrams)
        )
    else:
        bigram_diversity = 0

    return {
        "word_count": word_count,
        "unique_word_count": unique_word_count,
        "type_token_ratio": type_token_ratio,
        "avg_word_length": avg_word_length,
        "sentence_count": sentence_count,
        "avg_sentence_length": avg_sentence_length,
        "sentence_length_std": sentence_length_std,
        "max_sentence_length": max_sentence_length,
        "min_sentence_length": min_sentence_length,
        "filler_count": filler_count,
        "filler_ratio": filler_ratio,
        "repeated_word_count": repeated_word_count,
        "repetition_ratio": repetition_ratio,
        "stopword_count": stopword_count,
        "stopword_ratio": stopword_ratio,
        "pronoun_count": pronoun_count,
        "pronoun_ratio": pronoun_ratio,
        "comma_count": comma_count,
        "question_count": question_count,
        "exclamation_count": exclamation_count,
        "dependency_count": dependency_count,
        "subordinate_count": subordinate_count,
        "conjunction_count": conjunction_count,
        "bigram_diversity": bigram_diversity,
        "noun_ratio": pos_ratio("NOUN"),
        "verb_ratio": pos_ratio("VERB"),
        "adjective_ratio": pos_ratio("ADJ"),
        "adverb_ratio": pos_ratio("ADV"),
        "pronoun_pos_ratio": pos_ratio("PRON"),
        "determiner_ratio": pos_ratio("DET"),
        "preposition_ratio": pos_ratio("ADP"),
        "conjunction_pos_ratio": pos_ratio("CCONJ"),
        "auxiliary_ratio": pos_ratio("AUX"),
        "modal_ratio": pos_ratio("AUX"),
    }

In [ ]:
ling_rows = []

for text in tqdm(
    train_transcripts["transcript"],
    total=len(train_transcripts),
    desc="Extracting training linguistic features"
):
    ling_rows.append(extract_linguistic_features(text))

linguistic_df = pd.DataFrame(ling_rows)
linguistic_df = pd.concat(
    [
        train_transcripts[["filename", "label"]].reset_index(drop=True),
        linguistic_df.reset_index(drop=True),
    ],
    axis=1
)

print("Training linguistic features:", linguistic_df.shape)

In [ ]:
final_train_df = pd.merge(
    train_acoustic,
    linguistic_df,
    on=["filename", "label"],
    how="inner"
)

print("Final training dataframe:", final_train_df.shape)

X = (
    final_train_df
    .drop(columns=["filename", "label"], errors="ignore")
    .select_dtypes(include=[np.number])
)

y = final_train_df["label"].values

print("Feature matrix:", X.shape)
print("Target:", y.shape)
print("Numeric features:", X.shape[1])

## 6. Model Evaluation

The evaluation uses **5-fold cross-validation** with shuffling and a fixed random seed.
The primary metrics are:

- **RMSE** — lower is better.
- **Pearson correlation** — higher is better.

The model pipelines use median imputation so they can handle occasional missing acoustic values.

In [ ]:
def evaluate_model(model, X, y, model_name):
    kf = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    fold_rmse = []
    fold_pearson = []

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(X), 1
    ):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y[train_idx]
        y_val = y[val_idx]

        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        fold_rmse.append(
            np.sqrt(mean_squared_error(y_val, pred))
        )
        fold_pearson.append(
            pearsonr(y_val, pred)[0]
        )

    return {
        "model": model_name,
        "RMSE": np.mean(fold_rmse),
        "Pearson": np.mean(fold_pearson),
    }

models = {
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=5,
            loss="huber",
            random_state=42
        ))
    ]),
    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),
}

results = [
    evaluate_model(model, X, y, name)
    for name, model in models.items()
]

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

### Tuned Random Forest

The selected Random Forest configuration from the completed experimentation was:

- `n_estimators = 700`
- `max_depth = None`
- `min_samples_leaf = 3`
- `max_features = 0.5`
- `random_state = 42`

In [ ]:
rf_final_spec = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=700,
        max_depth=None,
        min_samples_leaf=3,
        max_features=0.5,
        random_state=42,
        n_jobs=-1
    ))
])

## 7. Out-of-Fold Predictions and Ensemble

The completed experiment generated out-of-fold predictions for the three candidate models.
The best blend was:

**50% Random Forest + 50% Ridge**

This blend achieved the strongest OOF RMSE/Pearson combination among the tested blends.

In [ ]:
oof_models = {
    "Random Forest": rf_final_spec,
    "Gradient Boosting": models["Gradient Boosting"],
    "Ridge": models["Ridge"],
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions = {
    name: np.zeros(len(y))
    for name in oof_models
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"Processing fold {fold}/5...")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y[train_idx]

    for name, model in oof_models.items():
        fitted = clone(model)
        fitted.fit(X_train, y_train)
        oof_predictions[name][val_idx] = fitted.predict(X_val)

In [ ]:
oof_results = []

for name, pred in oof_predictions.items():
    oof_results.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "Pearson": pearsonr(y, pred)[0],
    })

oof_results_df = (
    pd.DataFrame(oof_results)
    .sort_values("RMSE")
)

display(oof_results_df)

rf_oof = oof_predictions["Random Forest"]
ridge_oof = oof_predictions["Ridge"]

best_oof_pred = (
    0.50 * rf_oof
    + 0.50 * ridge_oof
)

final_oof_rmse = np.sqrt(
    mean_squared_error(y, best_oof_pred)
)
final_oof_pearson = pearsonr(
    y, best_oof_pred
)[0]

print(f"Final OOF RMSE: {final_oof_rmse:.6f}")
print(f"Final OOF Pearson: {final_oof_pearson:.6f}")

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y, best_oof_pred, alpha=0.6)

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    linestyle="--"
)

plt.xlabel("Actual Grammar Score")
plt.ylabel("Predicted Grammar Score")
plt.title("OOF Actual vs Predicted Grammar Scores")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Completed OOF Results

| Model / Blend | RMSE | Pearson |
|---|---:|---:|
| Random Forest | 0.733441 | 0.807596 |
| Gradient Boosting | 0.745761 | 0.798491 |
| Ridge | 0.747882 | 0.797986 |
| **RF 50% + Ridge 50%** | **0.714242** | **0.817797** |

## 8. Train Final Models on All Training Data

The final Random Forest and Ridge pipelines are fit on all 769 labeled samples before
test-set inference.

In [ ]:
final_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=700,
        max_depth=None,
        min_samples_leaf=3,
        max_features=0.5,
        random_state=42,
        n_jobs=-1
    ))
])

final_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0))
])

final_rf.fit(X, y)
final_ridge.fit(X, y)

print("Final models trained.")
print("Training samples:", len(X))
print("Features:", X.shape[1])

## 9. Test Transcription and Exact Feature Alignment

The same Whisper and feature-extraction functions are applied to the 216 test samples.

A key implementation detail is to construct the test features using **the exact same
acoustic and linguistic feature extractors as the training set**, then align the
columns to `X.columns`.

In [ ]:
if os.path.exists(TEST_TRANSCRIPT_PATH):
    test_transcripts = pd.read_csv(TEST_TRANSCRIPT_PATH)
    print("Loaded cached test transcripts:", test_transcripts.shape)
else:
    test_results = []

    for filename in tqdm(
        test_df["filename"],
        total=len(test_df),
        desc="Transcribing test audio"
    ):
        try:
            text = transcribe_audio(
                os.path.join(TEST_AUDIO_DIR, filename)
            )
            error = None
        except Exception as e:
            text = ""
            error = str(e)

        test_results.append({
            "filename": filename,
            "transcript": text,
            "error": error
        })

    test_transcripts = pd.DataFrame(test_results)
    test_transcripts.to_csv(TEST_TRANSCRIPT_PATH, index=False)

print("Test transcripts:", test_transcripts.shape)
print("Errors:", test_transcripts["error"].notna().sum())
display(test_transcripts.head())

In [ ]:
test_acoustic_path = os.path.join(ARTIFACT_DIR, "test_acoustic_features.csv")

if os.path.exists(test_acoustic_path):
    test_acoustic = pd.read_csv(test_acoustic_path)
else:
    test_acoustic_rows = []

    for filename in tqdm(
        test_df["filename"],
        desc="Extracting test acoustic features"
    ):
        features = extract_audio_features(
            os.path.join(TEST_AUDIO_DIR, filename)
        )
        features["filename"] = filename
        test_acoustic_rows.append(features)

    test_acoustic = pd.DataFrame(test_acoustic_rows)
    test_acoustic.to_csv(test_acoustic_path, index=False)

test_ling_rows = []

for text in tqdm(
    test_transcripts["transcript"],
    total=len(test_transcripts),
    desc="Extracting test linguistic features"
):
    test_ling_rows.append(
        extract_linguistic_features(text)
    )

test_linguistic = pd.DataFrame(test_ling_rows)
test_linguistic = pd.concat(
    [
        test_transcripts[["filename"]].reset_index(drop=True),
        test_linguistic.reset_index(drop=True),
    ],
    axis=1
)

test_features = pd.merge(
    test_acoustic,
    test_linguistic,
    on="filename",
    how="inner"
)

# Exact train/test feature alignment
train_feature_columns = X.columns.tolist()
X_test = test_features.reindex(
    columns=["filename"] + train_feature_columns
)

print("Test feature matrix:", X_test.shape)
print("Expected:", (216, 92))
print("Numeric features:", len(train_feature_columns))

missing = [
    c for c in train_feature_columns
    if c not in test_features.columns
]
extra = [
    c for c in test_features.columns
    if c != "filename" and c not in train_feature_columns
]

print("Missing features after exact extractor:", missing)
print("Extra features after exact extractor:", extra)

assert X_test.shape == (len(test_df), len(train_feature_columns) + 1)

In [ ]:
X_test_model = X_test[train_feature_columns].copy()

rf_pred = final_rf.predict(X_test_model)
ridge_pred = final_ridge.predict(X_test_model)

test_pred = (
    0.50 * rf_pred
    + 0.50 * ridge_pred
)

# Keep scores in the task's 0–5 range.
test_pred = np.clip(test_pred, 0, 5)

print("Predictions:", len(test_pred))
print("NaN predictions:", np.isnan(test_pred).sum())
print("Minimum:", float(test_pred.min()))
print("Maximum:", float(test_pred.max()))
print("Mean:", float(test_pred.mean()))

In [ ]:
SUBMISSION_PATH = os.path.join(ARTIFACT_DIR, "submission.csv")

submission = test_df[["filename"]].copy()
submission["label"] = test_pred
submission.to_csv(SUBMISSION_PATH, index=False)

print("Submission saved:", SUBMISSION_PATH)
display(submission.head(10))

In [ ]:
# Final submission integrity check

submission_check = pd.read_csv(SUBMISSION_PATH)

assert submission_check.shape == (len(test_df), 2)
assert list(submission_check.columns) == ["filename", "label"]
assert submission_check["filename"].equals(test_df["filename"])
assert submission_check["filename"].duplicated().sum() == 0
assert submission_check["label"].isna().sum() == 0
assert pd.api.types.is_numeric_dtype(submission_check["label"])
assert (
    (submission_check["label"] >= 0)
    & (submission_check["label"] <= 5)
).all()

print("✅ Submission is valid.")
print("Rows:", len(submission_check))
print("Columns:", list(submission_check.columns))
print("Label range:",
      submission_check["label"].min(),
      "to",
      submission_check["label"].max())

## 10. Recorded Kaggle Result

The first submitted iteration of this project achieved:

**Kaggle public score: 0.7326**

The challenge reports that the public leaderboard uses approximately 60% of the test data,
so the final score can differ.

The submitted approach was the same high-level multimodal pipeline documented here:
Whisper transcription + linguistic/acoustic features + 50/50 Random Forest/Ridge ensemble.

## 11. Limitations and Next Steps

- The dataset is private and cannot be redistributed with this repository.
- Whisper transcription quality can affect linguistic feature quality.
- The audio-derived feature set summarizes speech statistics rather than modeling full
  temporal sequences.
- Additional gains could come from stronger pretrained speech/language representations,
  but the current pipeline was intentionally kept interpretable and easy to explain.